# Notebook 02 — Filtrado y selección

¡Bienvenido al segundo notebook! 🐧

En el Notebook 01 aprendiste a **cargar e inspeccionar** un DataFrame. Ahora vas a aprender la operación más usada en data analysis: **quedarte solo con las filas que te interesan** (filtrar) y combinar eso con la selección de columnas.

## Objetivos de aprendizaje

Al terminar este notebook vas a poder:

1. Filtrar filas por una **condición booleana** (`df[df["col"] > x]`).
2. Combinar varias condiciones con **`&`** (Y) y **`|`** (O), respetando los paréntesis.
3. Filtrar por una lista de valores con **`.isin()`**.
4. Usar **`.loc[]`** para seleccionar **filas y columnas** a la vez.
5. **Ordenar** un DataFrame con **`.sort_values()`**.

Como antes: lee, ejecuta las celdas en orden, completa los ejercicios y verifica que los `assert` te den ✅. ¡Vamos!

---

## 1. Repaso rápido del Notebook 01

Recordatorio de lo que ya sabes hacer:

| Operación | Sintaxis |
|---|---|
| Cargar dataset | `sns.load_dataset("penguins")` |
| Ver primeras filas | `df.head()` |
| Forma | `df.shape` |
| Tipos | `df.dtypes` |
| Una columna (Series) | `df["species"]` |
| Varias columnas (DataFrame) | `df[["species", "island"]]` |
| Una fila por posición | `df.iloc[0]` |

Con esta base, vamos a sumar **condiciones**.

---

## 2. Setup — importar librerías y cargar el dataset

Volvemos a usar el dataset `penguins` para que puedas comparar resultados con el notebook anterior.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset("penguins")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---

## 3. Filtrar filas por una condición booleana

El patrón fundamental de filtrado en pandas tiene **dos pasos**:

1. **Construir una `Series` booleana** con la condición. Por ejemplo `df["body_mass_g"] > 5000` produce una Series con `True`/`False` por cada fila.
2. **Usar esa Series como índice**: `df[mask]` devuelve solo las filas donde la máscara es `True`.

### Demo paso a paso

Veamos primero cómo se ve la máscara booleana:

In [ ]:
# Step 1 — build a boolean mask (a Series of True/False)
heavy_mask = df["body_mass_g"] > 5000
heavy_mask.head(10)

In [ ]:
# Step 2 — use the mask to keep only matching rows
heavy_penguins = df[heavy_mask]
print(f"Penguins heavier than 5000g: {heavy_penguins.shape[0]} rows")
heavy_penguins.head()

👉 En la práctica, esto se suele escribir en **una sola línea**:

```python
df[df["body_mass_g"] > 5000]
```

Léelo de adentro hacia afuera: primero la condición, luego usarla para indexar.

### 🏋️ Ejercicio 3

Crea un DataFrame llamado **`adelie_df`** que contenga **solo los pingüinos de la especie `"Adelie"`**.

💡 Tip: la condición es `df["species"] == "Adelie"`.

In [ ]:
# YOUR CODE HERE
adelie_df = None


In [ ]:
# Tests — verify the filter is correct
assert isinstance(adelie_df, pd.DataFrame), "adelie_df must be a pandas DataFrame"
assert adelie_df.shape[0] == 152, f"Expected 152 Adelie penguins, got {adelie_df.shape[0]}"
assert (adelie_df["species"] == "Adelie").all(), "All rows must be of species 'Adelie'"

print(f"✅ ¡Filtro aplicado! Encontraste {adelie_df.shape[0]} pingüinos Adelie.")

---

## 4. Combinar condiciones con `&` (Y) y `|` (O)

Para combinar varias condiciones en pandas:

| Operador lógico | En Python normal | **En pandas** |
|---|---|---|
| Y | `and` | **`&`** |
| O | `or` | **`|`** |
| NO | `not` | **`~`** |

⚠️ **Regla de oro**: cada condición debe ir **entre paréntesis**. Si no, Python evalúa los operadores en orden incorrecto y te dará un error o un resultado equivocado.

### Demo: pingüinos Gentoo Y con peso mayor a 5000g

In [ ]:
# Combining two conditions with & — note the parentheses around each one
big_gentoo = df[(df["species"] == "Gentoo") & (df["body_mass_g"] > 5000)]
print(f"Big Gentoo penguins: {big_gentoo.shape[0]} rows")
big_gentoo.head()

In [ ]:
# Combining two conditions with | (OR)
small_or_chinstrap = df[(df["body_mass_g"] < 3500) | (df["species"] == "Chinstrap")]
print(f"Penguins that are small OR Chinstrap: {small_or_chinstrap.shape[0]} rows")

### 🏋️ Ejercicio 4

Crea un DataFrame llamado **`heavy_adelie_df`** que contenga los pingüinos que **a la vez** cumplen:

- Son de la especie `"Adelie"`, **Y**
- Tienen `body_mass_g` **mayor que** `4000`.

💡 Recuerda: usa `&` y pon **paréntesis** alrededor de cada condición.

In [ ]:
# YOUR CODE HERE
heavy_adelie_df = None


In [ ]:
# Tests
assert isinstance(heavy_adelie_df, pd.DataFrame), "heavy_adelie_df must be a DataFrame"
assert heavy_adelie_df.shape[0] == 35, f"Expected 35 rows, got {heavy_adelie_df.shape[0]}"
assert (heavy_adelie_df["species"] == "Adelie").all(), "All rows must be Adelie"
assert (heavy_adelie_df["body_mass_g"] > 4000).all(), "All rows must have body_mass_g > 4000"

print(f"✅ ¡Excelente! {heavy_adelie_df.shape[0]} pingüinos Adelie pesados encontrados.")

---

## 5. `.isin()` — filtrar por una lista de valores

Cuando quieres quedarte con filas cuya columna está dentro de **un conjunto de valores**, podrías escribir:

```python
df[(df["species"] == "Adelie") | (df["species"] == "Gentoo")]
```

Pero hay una forma más limpia: **`.isin()`**.

### Demo

In [ ]:
# Keep penguins of species Adelie or Gentoo (no need for chained ORs)
adelie_or_gentoo = df[df["species"].isin(["Adelie", "Gentoo"])]
print(f"Adelie or Gentoo: {adelie_or_gentoo.shape[0]} rows")
adelie_or_gentoo["species"].value_counts()

### 🏋️ Ejercicio 5

Crea un DataFrame llamado **`biscoe_or_dream_df`** con los pingüinos que viven en la isla `"Biscoe"` **o** la isla `"Dream"`.

Usa **`.isin()`** (no uses `|`).

In [ ]:
# YOUR CODE HERE
biscoe_or_dream_df = None


In [ ]:
# Tests
assert isinstance(biscoe_or_dream_df, pd.DataFrame), "biscoe_or_dream_df must be a DataFrame"
assert biscoe_or_dream_df.shape[0] == 292, f"Expected 292 rows, got {biscoe_or_dream_df.shape[0]}"
assert biscoe_or_dream_df["island"].isin(["Biscoe", "Dream"]).all(), "All islands must be Biscoe or Dream"
assert "Torgersen" not in biscoe_or_dream_df["island"].unique(), "Torgersen penguins should be excluded"

print(f"✅ ¡Listo! {biscoe_or_dream_df.shape[0]} pingüinos en Biscoe o Dream.")

---

## 6. `.loc[]` — seleccionar filas y columnas a la vez

Hasta ahora hicimos filtrado y selección de columnas en **dos pasos**:

```python
df[df["sex"] == "Female"][["species", "body_mass_g"]]
```

Esto funciona, pero pandas tiene una forma más clara y eficiente: **`.loc[filas, columnas]`**.

### Sintaxis

```python
df.loc[<condición de filas>, <lista de columnas>]
```

### Demo

In [ ]:
# Females, keeping only species and body_mass_g — all in one expression
female_subset = df.loc[df["sex"] == "Female", ["species", "body_mass_g"]]
female_subset.head()

### 🏋️ Ejercicio 6

Crea un DataFrame llamado **`gentoo_bills`** que contenga **únicamente** las columnas `"bill_length_mm"` y `"bill_depth_mm"`, y **únicamente** las filas de los pingüinos de la especie `"Gentoo"`.

Usa **`.loc[]`** en una sola expresión.

In [ ]:
# YOUR CODE HERE
gentoo_bills = None


In [ ]:
# Tests
assert isinstance(gentoo_bills, pd.DataFrame), "gentoo_bills must be a DataFrame"
assert gentoo_bills.shape == (124, 2), f"Expected shape (124, 2), got {gentoo_bills.shape}"
assert set(gentoo_bills.columns) == {"bill_length_mm", "bill_depth_mm"}, "Columns must be exactly bill_length_mm and bill_depth_mm"

print("✅ ¡Bien hecho! Filtraste filas y columnas en una sola operación con .loc.")

---

## 7. Ordenar con `.sort_values()`

Para **ordenar** un DataFrame por los valores de una columna se usa `.sort_values()`:

```python
df.sort_values("col")                    # ascendente (por defecto)
df.sort_values("col", ascending=False)   # descendente
df.sort_values(["col1", "col2"])         # ordenar por varias columnas
```

ℹ️ Por defecto los valores `NaN` van al **final**, sin importar la dirección. Eso es útil cuando hay datos faltantes.

### Demo

In [ ]:
# Smallest flippers first
smallest_flippers = df.sort_values("flipper_length_mm").head()
smallest_flippers

In [ ]:
# Heaviest penguin first
df.sort_values("body_mass_g", ascending=False).head(3)

### 🏋️ Ejercicio 7

Crea un DataFrame llamado **`top_10_heaviest`** con los **10 pingüinos de mayor `body_mass_g`**, ordenados de **mayor a menor**.

💡 Tip: combina `.sort_values(...)` con `.head(10)`.

In [ ]:
# YOUR CODE HERE
top_10_heaviest = None


In [ ]:
# Tests
assert isinstance(top_10_heaviest, pd.DataFrame), "top_10_heaviest must be a DataFrame"
assert top_10_heaviest.shape[0] == 10, f"Expected 10 rows, got {top_10_heaviest.shape[0]}"
assert top_10_heaviest.iloc[0]["body_mass_g"] == 6300, "The first row must be the heaviest penguin (6300 g)"
assert top_10_heaviest["body_mass_g"].is_monotonic_decreasing, "body_mass_g must be sorted descending"

print("✅ ¡Top 10 listo! Esos son los pingüinos más pesados del dataset.")

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Genial! Ahora dominas las operaciones de filtrado más usadas en data analysis:

| Operación | Sintaxis |
|---|---|
| Una condición | `df[df["col"] > x]` |
| Y lógico | `df[(c1) & (c2)]` |
| O lógico | `df[(c1) | (c2)]` |
| Negación | `df[~(c1)]` |
| Lista de valores | `df[df["col"].isin([...])]` |
| Filas + columnas | `df.loc[c1, ["col1", "col2"]]` |
| Ordenar ascendente | `df.sort_values("col")` |
| Ordenar descendente | `df.sort_values("col", ascending=False)` |

## ¿Qué viene en el próximo notebook?

En **Notebook 03 — Valores faltantes (NaN)** vas a aprender a:

- Detectar valores faltantes con `.isna()` y `.notna()`.
- Decidir qué hacer con ellos: **eliminar** filas (`.dropna()`) o **rellenar** con un valor (`.fillna()`).
- Entender cuándo conviene cada estrategia.

Notarás que el dataset `penguins` tiene algunos `NaN` que ignoramos en este notebook. ¡Los enfrentamos en el próximo! 🛠️